# TCC — Codificação de dados em classificadores quânticos variacionais

**A pergunta do trabalho:** como a escolha da codificação de dados clássicos em estados
quânticos afeta o desempenho e o custo de um classificador variacional? Quatro codificações
(*angle*, *amplitude*, *data re-uploading* e o *feature map* ZZ) são comparadas sob um
protocolo idêntico: ansatz, otimizador, dados e sementes ficam congelados em
`tccqml.config.PADRAO`, e só a codificação muda.

**Como usar.** Rode de cima a baixo. O notebook não reimplementa nada: cada experimento é uma
chamada a `python -m tccqml <subcomando>`, e o comando equivalente aparece numa linha que
começa com `$`. Por isso os números daqui são os mesmos do terminal e do texto. O código de
exibição fica em [`tccqml/notebook.py`](../src/tccqml/notebook.py), para que cada célula tenha
uma ou duas linhas.

Com `RODAR = False` (o padrão), o notebook só **lê** os resultados versionados em `results/`
e roda inteiro em cerca de um minuto. Com `RODAR = True`, roda os experimentos cujos arquivos
faltarem (do zero, cerca de uma hora numa CPU comum). O treino isolado da seção 3 roda
sempre: é uma ação explícita e leva segundos.

Para só **ler**, sem instalar nada, existe [`tcc_qml_executado.ipynb`](tcc_qml_executado.ipynb):
este mesmo notebook já executado, gerado por `python notebooks/gerar_executado.py`.

| Seção | O que tem |
|---|---|
| 0. Setup | opções, versões e o que está registrado na CLI |
| 1. Os dados | os três conjuntos, bruto × normalizado, partições e as hipóteses do texto |
| 2. Anatomia | custo, circuito e estado $\psi(x)$ de cada codificação |
| 3. Um treino | uma combinação qualquer, com curvas e fronteira de decisão |
| 4. Os experimentos | um subcomando por seção, com as tabelas e figuras do TCC |
| 5. Índice | onde cada tabela e figura entra no texto |

## 0. Setup

In [ ]:
from tccqml import notebook as nb

RODAR = False  # True: roda os experimentos cujos arquivos faltarem (do zero, ~1 h)
nb.iniciar(rodar=RODAR)

O `listar` mostra o que está registrado (codificações, conjuntos, ansätze) e o protocolo
congelado. Para rodar a suíte de testes daqui, use `nb.testes()` (cerca de 2 minutos); se ela
falhar, nenhum número abaixo deve ser usado.

In [ ]:
nb.tccqml("listar");

## 1. Os dados

Nada nesta seção é quântico: são pontos no plano com rótulo 0 ou 1, gerados a partir de uma
semente (não há arquivo de dados no repositório). Cada conjunto tem $N = 300$ amostras com
ruído 0,15, dividido de forma estratificada em treino, validação e teste (60/20/20) e
normalizado para $[0, \pi]$. A figura é a `datasets.pdf` do comando `figuras`.

In [ ]:
nb.mostrar_figura("datasets.pdf")

### As hipóteses do texto

São as registradas no Cap. 3 ("Previsões e hipóteses", Tabela `tab:hipoteses`), todas nas
coordenadas já normalizadas. Elas tratam do **suporte** de frequências permitido, que não
garante que o circuito realize os coeficientes necessários nem que o treino os encontre.

- **H1, moons**: fronteira aberta que muda de concavidade. O *angle* teria desempenho
  limitado, e o *re-uploading* melhoraria a partir de $R = 2$. O limiar $R = 2$ foi registrado,
  mas não demonstrado para este circuito, e a previsão é tratada como pergunta experimental.
- **H2, circles**: $g(x) = \sin x_1 + \sin x_2$, de frequência 1, tem curvas de nível fechadas
  em torno do centro de $[0, \pi]^2$, então *angle* e *re-uploading* não se diferenciariam. A
  seção 4.8 verifica se um limiar sobre $g$ separa de fato as classes.
- **H3, xor**: a classe depende do sinal do produto dos atributos originais. O termo cruzado
  $\cos x_1 \cos x_2$ do *angle* tem esse padrão em torno de $(\pi/2, \pi/2)$, e com a leitura
  local de $Z_0$ ele só chega à saída pelas CNOTs do ansatz.
- **H4, amplitude**: com os dados em $[0, \pi]^2$, a fronteira é formada por direções que
  partem da origem, o que não separa *xor* nem *circles*.
- **H5, saturação**: o ganho do *re-uploading* para de crescer a partir de certo $R$, enquanto
  o custo cresce linearmente.

Nenhuma hipótese foi registrada para o *feature map* ZZ: os resultados dele são exploratórios.

### Bruto × normalizado

O reescalonamento é feito **por atributo**, com os extremos do treino. Por isso ele não
preserva necessariamente a forma das figuras (escalas diferentes nos dois eixos transformam
círculos em elipses), e a origem dos dados originais só vai para $(\pi/2, \pi/2)$ quando os
extremos de cada atributo são simétricos. À esquerda, os pontos $u$ do gerador; à direita, o
treino $x$ depois do `MinMaxScaler`, com linhas pontilhadas em $0$ e $\pi$.

In [ ]:
nb.bruto_vs_normalizado("moons")  # ou "xor", "circles"

### Partições

O treino tem mínimo $0$ e máximo $\pi$ exatos. Validação e teste podem sair **um pouco fora**
de $[0, \pi]$, e isso é proposital: o `MinMaxScaler` é ajustado **só no treino**. Ajustar no
conjunto inteiro faria os limites baterem, mas vazaria informação da validação e do teste para
o treino e inflaria a acurácia.

In [ ]:
nb.particoes()

### Por que $[0, \pi]$

O intervalo é uma escolha de **escala e localidade**, e não uma condição para não perder
informação. No *angle*, o *kernel* induzido é $k(x, x') = \prod_i \cos^2\big((x_i - x_i')/2\big)$.
Com os atributos de treino em $[0, \pi]$, a diferença entre dois valores de um atributo fica em
$[0, \pi]$, trecho em que cada fator decresce de 1 a 0: valores próximos viram estados
parecidos, e os extremos, estados ortogonais. Um intervalo de largura $2\pi$ levaria os dois
extremos ao **mesmo estado físico**, porque $R_Y(x + 2\pi) = -R_Y(x)$. A seção 4.7
(`diagnostico`) mostra o que acontece com o *amplitude* quando a normalização muda.

## 2. Anatomia do classificador

A cadeia completa, a mesma para as quatro codificações:

$$x \;\mapsto\; \psi(x) \;\mapsto\; U(\theta)\,\psi(x) \;\mapsto\; \langle Z_0 \rangle + b \;\mapsto\; \hat{y}$$

O bloco de dados $x \mapsto \psi(x)$ é a codificação, a única parte que varia. $U(\theta)$ é o
ansatz congelado (`StronglyEntanglingLayers`, $L_\text{var} = 2$ camadas). A medição de $Z_0$ e
o viés clássico $b$ são iguais para todos.

A célula abaixo monta o circuito com `tccqml.model.build`, a mesma função que a CLI usa, com os
pesos iniciais da semente do protocolo, e mostra: o **bloco de custo** que o `treinar` imprime,
com a contagem de *parameter-shift* ($2p + 1$ avaliações por amostra, $(2p + 1)|B|$ por passo e
$(2p + 1)|B|S$ shots por passo); o **circuito**, em blocos e com `level="device"` (as portas
que a máquina executaria); e o **estado $\psi(x)$** só do bloco de dados, para
$x = (0{,}5;\ 2{,}0)$. Troque o argumento para ver as outras codificações (no Jupyter ou no VS
Code, `nb.painel_anatomia()` abre um seletor).

In [ ]:
nb.anatomia("angle")  # angle | amplitude | reuploading | zz

### Como ler o desenho de cada codificação

- **`angle`**: `RY(0.50)` e `RY(2.00)` *são* os dados, um qubit por atributo. O bloco de dados
  tem profundidade 1 e nenhuma CNOT. $p = 12$ (2 camadas × 2 qubits × 3 ângulos do `Rot`).
- **`amplitude`**: os dois atributos viram as duas amplitudes de **um único qubit**, depois de
  normalizados. Com um qubit não há CNOT, e $p$ cai de 12 para **6**. A normalização descarta
  a norma de $x$: dois pontos na mesma direção viram o mesmo estado.
- **`reuploading`**: o mesmo `RY` do *angle*, repetido $R = 3$ vezes. O
  ansatz **não vem depois**: uma camada treinável fica depois de cada bloco de dados, e por
  isso $p = 6R = $ **18**.
- **`zz`**: `Hadamard`, `RZ(2 x_i)` e, para o par, `CNOT · RZ · CNOT`, repetidos
  $r_{\mathrm{ZZ}} = 2$ vezes. Entre as implementadas, com $d = 2$, é a **única com CNOTs no
  próprio bloco de dados**: as 4 CNOTs do bloco somam-se às 4 do ansatz.

**Estado produto × entrelaçado.** A concorrência diz se o bloco de dados já entrelaça. No
*angle* e no *re-uploading* o bloco é feito de rotações independentes por qubit, então o estado
é **produto** e o entrelaçamento só aparece no ansatz. No *amplitude* não aparece em lugar
nenhum, porque há um qubit só. No *zz* o estado **pode** sair entrelaçado já do bloco de dados,
antes de qualquer parâmetro treinável: os termos $Z_iZ_j$ fazem o estado depender de pares de
atributos já na codificação.

**Retropropagação no simulador × *parameter-shift* contado.** O treino calcula o gradiente por
retropropagação sobre o simulador, o caminho mais barato em software. O *parameter-shift*, que
valeria em hardware real, **não é executado**: é contado analiticamente. O gradiente do custo
quadrático é $2\,(f(x) + b - y)\,\partial f/\partial\theta$; as $2p$ avaliações deslocadas dão
$\partial f/\partial\theta$, e o resíduo pede mais uma, sem deslocamento: $2p + 1$ por amostra.
O viés $b$ não entra, porque sua derivada é clássica.

**O espectro previsto.** Quando os dados entram linearmente nos ângulos de rotações de Pauli, a
saída é uma série de Fourier truncada, $f_\theta(x) = \sum_{\omega \in \Omega} c_\omega(\theta)\,
e^{i\omega x}$ (Schuld et al., 2021). Quem fixa o suporte $\Omega$ é a **codificação**; o ansatz
e o observável determinam quais coeficientes são realizáveis. No *angle*,
$\Omega = \{-1, 0, 1\}$ por atributo; no *re-uploading* com $R$ repetições,
$\Omega = \{-R, \dots, R\}$. Para *amplitude* e *zz* a saída não é uma série de Fourier em $x$.

## 3. Um treino isolado

A célula roda `python -m tccqml treinar ... --salvar` com `--out results/notebook`, para não
tocar nos resultados oficiais, e mostra o custo, as curvas, a fronteira de decisão e a
comparação com a grade. As opções, com os padrões do protocolo, são `ansatz` (`"local"` é o
ansatz sem CNOTs da ablação), `R` (só no *re-uploading*), `L_var`, `epocas`, `eta`
e `seed`; por exemplo, `nb.treinar("reuploading", "moons", R=5)`. No Jupyter ou no VS
Code, `nb.painel_treino()` abre as mesmas opções num painel com botão.

**A divisão 60/20/20.** As 300 amostras viram 180 de treino, 60 de validação e 60 de teste. O
treino ajusta os pesos; a **validação escolhe a época de parada** (a de maior acurácia de
validação; no empate, a mais antiga); o **teste é medido uma única vez**, nesse modelo, e nunca
aparece nas curvas. É também por isso que a validação costuma sair um pouco acima do teste:
ela é o máximo de uma curva ruidosa.

Com a configuração do protocolo e uma semente da grade, o resultado é **idêntico** ao da linha
correspondente de `comparar`, e a célula confere isso no fim. Mudar camadas, épocas ou `eta` sai
do protocolo, e a célula avisa quando a comparação com a grade deixa de valer.

In [ ]:
nb.treinar("angle", "moons")  # encoding: angle | amplitude | reuploading | zz; dataset: xor | moons | circles

**O que experimentar:** `angle` contra `reuploading` no `moons` é o par da hipótese H1; no
`circles`, H2 prevê que os dois não se diferenciem. Com `ansatz="local"` no `xor`, o modelo
perde as CNOTs (H3).

## 4. Os experimentos do TCC

Uma seção por subcomando, na ordem do texto. Cada célula roda o experimento se faltar algum
arquivo dele e `RODAR = True` (e, nesse caso, regenera tabelas e figuras); depois mostra as
tabelas de `results/tables/`, as mesmas que vão para o Overleaf, e as figuras de
`results/figures/`. Os tempos citados são de treino numa CPU comum, sem GPU.

### 4.1 `comparar`: a grade principal

**Pergunta:** com tudo o mais congelado, qual codificação classifica melhor cada conjunto, e a
que custo? Testa H2 e H4 e dá os resultados exploratórios do ZZ.

**O que faz:** treina 4 codificações × 3 conjuntos × 5 sementes (60 treinos, cerca de 7 minutos)
e grava `metrics/comparacao.csv` (uma linha por treino e época), `metrics/por_treino.csv`,
`metrics/resumo.csv` e os pesos em `weights/`.

**O que procurar:** leia a `tab_acuracia` conjunto a conjunto; uma diferença menor que o desvio
entre sementes **não** sustenta uma ordem. Depois cruze com a `tab_custo` e a
`acuracia-vs-custo`, porque a comparação é de acurácia **e** custo: profundidade, CNOTs e shots
por passo. As fronteiras explicam as acurácias, e as curvas mostram se 30 épocas bastaram.

In [ ]:
nb.experimento("comparar")

### 4.2 `sensibilidade`: a taxa de aprendizado

**Pergunta:** quando uma codificação fica para trás, é a representação ou só o passo comum
$\eta = 0{,}1$?

**O que faz:** repete a grade variando **só** $\eta$ sobre 0,01, 0,03, 0,1 e 0,3, fixados
antes de rodar, com o mesmo orçamento para todas as codificações (240 treinos, cerca de quatro vezes o
tempo da grade). A taxa é
escolhida pela maior acurácia média de **validação**; o teste não participa.

**O que procurar:** se a acurácia de teste com o $\eta$ escolhido difere da do protocolo menos
que o desvio, a codificação não estava limitada pela taxa de aprendizado. Um ganho na validação
que não aparece no teste é em parte efeito da própria escolha.

In [ ]:
nb.experimento("sensibilidade")

### 4.3 `varredura --parametro R`: quantas repetições no re-uploading

**Pergunta:** a acurácia do *re-uploading* cresce com o número $R$ de repetições, e até
onde? (H1 e H5)

**O que faz:** treina o *re-uploading* com $R = 1, \dots, 5$ nos três conjuntos (75 treinos,
cerca de 13 minutos). O custo cresce linearmente: $p = 6R$.

**O que procurar:** onde cada curva **estabiliza**; a partir daí, mais $R$ só custa. Uma queda
com $R$ só é lida como sobreajuste se as curvas de treino e validação a sustentarem. $R = 1$ no
nível do acaso no *xor* não é falta de frequência: com o par de CNOTs e a medição de $Z_0$, o
último bloco de dados só alcança $x_2$ (seção 4.5). Com $R$ crescem **ao mesmo tempo** o
suporte e os parâmetros; a seção seguinte separa os dois efeitos.

In [ ]:
nb.experimento("varredura_R")

### 4.4 `varredura --parametro L_var`: controle com o mesmo $p$

**Pergunta:** o ganho do *re-uploading* vem das frequências acessíveis ou só do número de
parâmetros?

**O que faz:** treina o *angle* com $L_\text{var} = k = 1, \dots, 5$ camadas (75 treinos, cerca de 12 minutos). Ele
tem o mesmo $p = 6k$ do *re-uploading* com $R = k$, mas continua com $\Omega = \{-1, 0, 1\}$. É
uma exceção deliberada ao protocolo, gravada num CSV próprio.

**O que procurar:** leia a `tab_mesmo_p` linha a linha. Com $k = 1$ os dois circuitos são
idênticos, e com $k = 2$ o *angle* é o da grade principal, o que serve de conferência. Se o
*angle* fica parado enquanto o *re-uploading* sobe, o ganho vem da **reinserção dos dados**, e
não do número de parâmetros; a profundidade, porém, não é a mesma nos dois.

In [ ]:
nb.experimento("varredura_L_var")

### 4.5 `espectro`: $\Omega$ medido

**Pergunta:** as frequências que o modelo de fato representa respeitam o suporte previsto?

**O que faz:** avalia a saída numa grade de $32 \times 32$ pontos de $[0, 2\pi)^2$ e conta, por
FFT 2D, os termos $(\omega_1, \omega_2)$ com energia, para o *angle* e o *re-uploading* com
$R = 1, 2, 3$. Não treina nada (segundos).

**O que procurar:** $\Omega$ é um **limite superior**. A coluna `dentro_do_limite` verifica que
nenhuma energia escapa dele, e `atinge_limite` diz se ele é alcançado. No *re-uploading*, $x_1$
para em $R - 1$, porque o par de CNOTs e a medição local de $Z_0$ desperdiçam o último bloco de
dados nesse atributo: uma perda da arquitetura, não da codificação.

In [ ]:
nb.experimento("espectro")

### 4.6 `ablacao`: o ansatz sem CNOTs

**Pergunta:** o classificador depende do entrelaçamento do ansatz? (H3)

**O que faz:** treina o *angle* com o ansatz `local` (as mesmas rotações e o mesmo $p = 12$, sem
CNOTs) nos três conjuntos, ao lado do ansatz padrão (30 treinos, cerca de 3 minutos).

**O que procurar:** sem CNOTs, o estado é produto e $\langle Z_0\rangle$ depende só de $x_1$:
espera-se queda em **todos** os conjuntos e o *xor* no nível do acaso. A conclusão vale **dado o
observável local $Z_0$**: com o mesmo estado produto, medir $Z_0 Z_1$ daria $\cos x_1 \cos x_2$,
cujo sinal reproduz o padrão do *xor*.

In [ ]:
nb.experimento("ablacao")

### 4.7 `diagnostico`: o amplitude e a normalização

**Pergunta:** o desempenho do *amplitude* é da codificação ou da normalização $[0, \pi]$? (H4)

**O que faz:** treina o *amplitude* com os dados em $[0, \pi]$ (o protocolo) e em $[-1, 1]$
(centrados na origem), nos três conjuntos (30 treinos, cerca de 2 minutos).

**O que procurar:** o *amplitude* só vê a **direção** de $x$. Em $[-1, 1]$ a origem fica no
centro dos dados, e as classes do *xor*, separadas por quadrantes, ficam ao alcance de uma
fronteira radial; *moons* e *circles* podem perder a estrutura. A comparação principal mantém
$[0, \pi]$ para todas as codificações, e o diagnóstico mostra o preço dessa escolha.

In [ ]:
nb.experimento("diagnostico")

### 4.8 `verificacoes`: o que os próprios dados permitem

**Pergunta:** os limites observados são dos circuitos ou dos dados?

**O que faz:** duas checagens clássicas, sem circuito, com as mesmas partições e sementes da
grade (segundos):

- **controle linear**: regressão logística ($C = 1$, até 1000 iterações) sobre as nove funções
  $\{1, \cos x_j, \sin x_j\}$ e seus produtos, o espaço de frequência 1 do *angle*;
- **limiar de $g$ no circles**: a regra $g(x)/2 > t$, com $g(x) = \sin x_1 + \sin x_2$ e o
  limiar escolhido no treino, sem ruído e com o ruído do protocolo.

**O que procurar:** se o controle linear empata com o *angle*, o teto do *angle* **pode** vir do
espaço de funções, e não do treino; a seção 4.2 e a `tab_mesmo_p` são as outras evidências a
cruzar. O controle acerta o que o espaço *contém*, não o que o circuito consegue realizar ou
aprender. No *circles*, compare o limiar sem e com ruído.

In [ ]:
nb.experimento("verificacoes")

## 5. Índice: onde cada tabela e figura entra no TCC

`tabelas` grava `.tex` (colável no Overleaf) e `.csv` em `results/tables/`; `figuras` grava os
PDFs vetoriais em `results/figures/`. Os dois só leem CSV e nunca treinam. Os rótulos `tab:`
são os dos `.tex`; os `fig:` são sugestões, a fixar quando o Cap. 4 for escrito. A única tabela
que não apareceu acima é a `tab_qualitativa`, que é julgamento do autor e não medição.

In [ ]:
nb.indice()
nb.mostrar_tabela("tab_qualitativa")

## 6. A sequência completa no terminal

Da raiz do repositório, com o ambiente ativado:

```bash
pip install -r requirements-lock.txt
pip install -e .
pytest

python -m tccqml comparar
python -m tccqml sensibilidade
python -m tccqml varredura --parametro R
python -m tccqml varredura --parametro L_var
python -m tccqml espectro
python -m tccqml ablacao
python -m tccqml diagnostico
python -m tccqml verificacoes
python -m tccqml tabelas
python -m tccqml figuras
```